> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# **IS 4487 LAB 11: Linear Regression for HR**

In this lab, we'll use an HR dataset to build a **linear regression model** to predict the number of years that an employee will work for the company (job tenure).


## Outline
1. Load and explore the dataset
2. Clean and prepare features
3. Encode categorical variables
4. Split the data into training and test sets
5. Train and evaluate a linear regression model
6. Reflect on variable importance and model fit




<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Labs/lab_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# **Context: HR data**

## Problem Statement
A large company employs, at any given point of time, around 4000 employees. However, every year, around 15% of its employees leave the company and need to be replaced with the talent pool available in the job market. The management believes that this employee turnover is bad for the company, because of the following reasons -

- The former employees' projects get delayed, which makes it difficult to meet timelines, resulting in a reputation loss among consumers and partners
- A sizeable department has to be maintained, for the purposes of recruiting new talent
- More often than not, the new employees have to be trained for the job and/or given time to acclimatize themselves to the company

Hence, the management has contracted an HR analytics firm to understand what factors they should focus on, in order to increase the number of years that employees stay with the company. In other words, they want to know what changes they should make to their workplace, in order to get most of their employees to stay. Also, they want to know which of these variables is most important and needs to be addressed right away.

Since you are one of the star analysts at the analytics firm, this project has been given to you.

## Goal of the case study
You are required to model the number of years employees work for the company using a linear regression model. The results thus obtained will be used by the management to understand what changes they should make to their workplace, in order to get most of their employees to stay.


## Dataset Overview

**Dataset:** `merged_hr_data.csv`  
Source: [Kaggle HR Analytics Case Study](https://www.kaggle.com/datasets/vjchoudhary7/hr-analytics-case-study)

| Variable                      | Type        | Description |
|-------------------------------|-------------|-------------|
| `Age`                         | Numeric     | Age of the employee |
| `Attrition`                   | Categorical | Whether the employee has left the company (Yes/No) |
| `BusinessTravel`              | Categorical | Frequency of business travel |
| `Department`                  | Categorical | Department name |
| `DistanceFromHome`           | Numeric     | Distance from home to work (in km) |
| `Education`                  | Ordinal     | Employee education level (1–5) |
| `EducationField`             | Categorical | Field of education |
| `EmployeeCount`              | Constant    | Always 1 (not useful for modeling) |
| `EmployeeID`                 | Identifier  | Unique identifier for employee |
| `EnvironmentSatisfaction`    | Ordinal     | Satisfaction with the environment (1–4) |
| `Gender`                     | Categorical | Gender of the employee |
| `JobInvolvement`             | Ordinal     | Level of involvement with job (1–4) |
| `JobLevel`                   | Ordinal     | Employee level (1–5) |
| `JobRole`                    | Categorical | Job title |
| `JobSatisfaction`            | Ordinal     | Satisfaction with the job (1–4) |
| `MaritalStatus`              | Categorical | Marital status |
| `MonthlyIncome`              | Numeric     | Monthly salary in USD |
| `NumCompaniesWorked`         | Numeric     | Number of companies previously worked for |
| `Over18`                     | Constant    | Always "Y" (not useful) |
| `PercentSalaryHike`          | Numeric     | Percentage salary increase |
| `PerformanceRating`          | Ordinal     | Performance rating (1–4) |
| `StandardHours`              | Constant    | Always 8 (not useful) |
| `StockOptionLevel`           | Ordinal     | Stock options level (0–3) |
| `TotalWorkingYears`          | Numeric     | Total years of professional experience |
| `TrainingTimesLastYear`      | Numeric     | Number of training sessions attended last year |
| `WorkLifeBalance`            | Ordinal     | Work-life balance rating (1–4) |
| `YearsAtCompany`             | Numeric     | Years spent at the current company |
| `YearsSinceLastPromotion`    | Numeric     | Years since last promotion |
| `YearsWithCurrManager`       | Numeric     | Years with current manager |

**Target Variable:** `YearsAtCompany`

## **Part 1: Import packages, load and preview data**




In [ ]:
import pandas as pd

# Load the merged HR dataset
url = "https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/merged_hr_data.csv"
df = pd.read_csv(url)

# this is a wide dataset with many columns - increase display width to show all
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)


# Preview structure
print("Shape:", df.shape)
df.head()


## **Part 2: Data Cleaning**

Real-world HR data often contains administrative fields (e.g., ID numbers), constants (same value for all rows), or missing values.

### What We’re Doing:
- Remove irrelevant or constant columns: `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeID`
- Drop rows with missing data

### Why It Matters:
- Non-informative or redundant features can reduce model accuracy and interpretability.
- Regression does not handle missing values natively, so we need a clean dataset.
- Dropping some rows is reasonable here due to the relatively small number of nulls.

> Ethical Note: In practice, dropping rows may disproportionately exclude certain groups—so this step should be handled with caution.


In [ ]:
# Drop unnecessary columns
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeID', 'Attrition']
df.drop(columns=drop_cols, inplace=True)

# Drop rows with any missing values
df.dropna(inplace=True)

# Check result
print("After cleaning:", df.shape)


## **Part 3: Encode Categorical Variables**

### Nominal variables

Machine learning algorithms like linear regression require **numeric inputs**. To use nominal categorical data like `Gender` or `JobRole` (where order does not matter), we convert them into **dummy variables** using **one-hot encoding.**

Example:
Colors   |  col_Blue | col_Green  | col_Red    |
|--------|-------|--------|--------|
Blue     |   1   | 0      | 0      |
Green    |  0    | 1      | 0      |
Red      | 0     | 0      | 1      |

When a categorical variable is dummy coded, we should drop (any) one of the dummy variables in order to prevent **multicollinearity** which occurs when one of the column is a perfect linear combination of one or more other columns in the data. The Colors column is perfectly described by the sum of the three dummy columns.

This is perfect multicollinearity—the model cannot uniquely estimate the coefficients because the columns are perfectly redundant.

> SOLUTION: drop one dummy coded column.

### Ordinal/ ordered variables

Ordered variables should be integer encoded to make them numeric. For example, Education, JobInvolvement, JobLevel and others that are ordinal are already integer coded in this dataset, so we do not need to do anything more. These variables will be treated like numeric data.


### Key Steps:
- first **integer encode** all ordinal columns and drop the original ordinal variable if present. In our data, this step has already been done. Else, you can use sklearn's `OrdinalEncoder` to fit and transform data
- second, perform **dummy variable encoding** using  `pd.get_dummies()` with `drop_first=True` to avoid multicollinearity. This function will convert all categorical columns (in our data all of which are nominal), into dummy variables.

### Why It Matters:
- Ensures model can interpret nominal categorical inputs numerically
- Dropping the first dummy prevents the **dummy variable trap** where one variable is a linear combination of others

> Reminder: Avoid encoding identifiers or columns with too many unique levels without reduction. Identifiers have no menainful relationship with outcomes. Too many dummy variables created due to having too many values in a caegorical variable will lead us to the other problem of **overfitting**



In [ ]:
# One-hot encode all categorical features (here ones remaining are all nominal)
df_encoded = pd.get_dummies(df, drop_first=True)

# Preview encoded columns
display(df_encoded.head())
df_encoded.info()

### **🔧 Try It Yourself - Part 3**

3.1. How many new columns were created during one-hot encoding?  you can compare the shape of the two dataframes in the outputs above.

3.2. Why is it important to avoid including columns like `EmployeeID` as a feature in predictive modeling?

3.3. Our model is trying to predict employment longevity.  Why is `Attrition` problematic for predicting years at the company?

Write a few sentences on each of the questions above. No coding is required here.


🔧 3.1 Add comment here:


🔧 3.2 Add comment here:


🔧 3.3 Add comment here:





### **Part 4: Standardizing Features for Regression**

When using models like **linear regression**, it's highly recommended to ensure all numeric features are on a similar scale. This helps the model converge more reliably and prevents features with larger magnitudes from dominating the learning process.

#### What data types to standardize?
- continuous: YES
- ordinal variables that are integer encoded: YES
- nominal categorical that are dummy coded/one-hot encoded: NO
- target variable: NO

In this step, we'll use `StandardScaler` from `sklearn` to scale all nuemric (continuous and integer-encoded feature) columns to have a mean of 0 and a standard deviation of 1. We do not scale the categorical features, as it would make their interpretation difficult.

This is especially important if your dataset includes variables with vastly different units or scales (e.g., "Age" vs. "MonthlyIncome")

> **Note:** The target variable (`YearsAtCompany`) should **not** be scaled — only the continuous input features.

---


In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate the features and the target
X_unscaled = df_encoded.drop(columns=['YearsAtCompany'])
y = df_encoded['YearsAtCompany']

# Identify columns that are truly continuous numeric features (excluding the target variable 'YearsAtCompany')
X_needs_scaling = X_unscaled.select_dtypes(include=['int', 'float'])

# Apply standardization to the continuous features (leave out all categorical encoded features)
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_needs_scaling)

# Reconstruct scaled DataFrame from the output of StandardScaler
X_numeric_scaled_df = pd.DataFrame(X_numeric_scaled, columns=X_needs_scaling.columns, index=df_encoded.index)

# combine this above DF with the DF of boolean columns selected from X_unscaled
X_boolean = X_unscaled.select_dtypes(include=['bool'])
X = pd.concat([X_numeric_scaled_df, X_boolean], axis=1)

# Preview the final set of features
X.head()

### **🔧 Try It Yourself - Part 4**

You've now scaled your features using `StandardScaler`, which makes each feature have a mean of 0 and a standard deviation of 1.

**Think about this:**
Suppose we didn't standardize the features and trained a regression model using raw input data instead. What might happen to the interpretation or relative importance of the coefficients?

4.1 Write one or two sentences explaining how not standardizing the data could affect the model's performance or interpretability.


🔧 4.1 Add comment here:

## **Part 5: Train-Test Split**

We'll split the dataset into:
- 80% for training
- 20% for testing

To preserve class proportions, we **stratify on our target variable**. This ensures fair evaluation.

> This step helps avoid training/test imbalance especially in classification tasks.



In [ ]:
from sklearn.model_selection import train_test_split

# Use already standardized features in X, and original target y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Check the shapes of the splits
X_train.shape, X_test.shape



### **🔧 Try It Yourself - Part 5**

5.1. In the code cell below, calculate what average `YearsAtCompany` is for all employees


In [ ]:
# 🔧 5.1 Add code here

5.2. Answer the following question in the markdown cell: Why is stratified sampling essential for classification with imbalanced target data?


🔧 5.2. Add your comment here

## **Part 6: Train the Regression**

Now we fit a linear regression model using the training data.  

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Display coefficients in order of highest to lowest correlation
coefficients_linear = pd.Series(linear_model.coef_, index=X.columns)
print("\nLinear Regression Coefficients (ordered):")
print(coefficients_linear.sort_values(ascending=False))


### **🔧 Try It Yourself - Part 6**

Write a few sentences on each of the questions below. No coding is required here.

6.1. Which features are most positively associated with high job tenure (years at company)?

6.2. Which features are most negatively associated with staying?




🔧 6.1. Add commment here


🔧 6.2. Add commment here



## **Part 7: Evaluate Model Performance**

Let's test how well our model generalizes to unseen data. We'll compute:
- Mean Squared Error (MSE)
- R-Squared

In [ ]:
# Make predictions on the test set
y_pred_linear = linear_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

# Print the evaluation metrics
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Now visualize the model output

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot of predicted vs actual YearsAtCompany
plt.figure(figsize=(10, 6))
ax = sns.regplot(x=y_test, y=y_pred_linear, scatter_kws={'alpha':0.6}, line_kws={"color": "red"})
ax.set_xlabel("Actual YearsAtCompany")
ax.set_ylabel("Predicted YearsAtCompany")
plt.title("Actual vs. Predicted YearsAtCompany with Regression Line")
plt.grid(True)
plt.show()

### **🔧 Try It Yourself - Part 7**

7.1. Is this R-squared fit good?

7.2. How could we improve the fit?

🔧 7.1. Add your comment here


🔧 7.2. Add your comment here

## **Part 8: Feature Selection for Performance Improvement**

Not all features equally influence `YearsAtCompany`. By identifying and using only the most important predictors, we can:
- Simplify the model, which **reduces overfitting**
- trade less variance for slightly more bias (recall **bias-variance tradeoff**)
- Potentially improve performance or interpretability


We'll use the linear regression model's coefficients to rank feature importance. To do so, we will use their absolute values (because we care about the magnitude of the coefficient, be it positive or negative).


In [ ]:
# Get the top 10 features based on the absolute coefficient magnitude
top_10_feature_names = coefficients_linear.abs().sort_values(ascending=False).head(10).index

# Print the actual coefficient values for these top 10 features
print("Top 10 features by absolute coefficient magnitude (with actual values):")
display(coefficients_linear.loc[top_10_feature_names])

### **🔧 Try It Yourself - Part 8**

8.1. Create a new training and test set using only the 10 most important features.

8.2. Retrain the linear regression model on this reduced dataset.

8.3. Evaluate performance of the new version


In [ ]:
# 🔧 Step 1: Create new versions of X_train and X_test with only those top features

# 🔧 Step 2a: Initialize and fit a new Regression model on the reduced feature set

# 🔧 Step 2b: Use the new model to predict on the test set

# 🔧 Step 3a: Create a chart to visualize the new model

# 🔧 Step 3b: Evaluate the reduced model using R-squared

## **🔧 Part 9: Reflection**

Write a few sentences on each of the questions below. No coding is required here.

9.1. How did the reduced-feature model compare to the full model?

9.2. Would this version be easier to explain or use in an HR meeting?



🔧 9.1. Add your comment here

🔧 9.2. Add your comment here

## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_11_LastnameFirstname.ipynb"